# BLOQUE 0: Repaso de la Clase 06 — Estadística y Preprocesamiento

Antes de meternos con IA, Machine Learning y Scikit-Learn, vamos a hacer un ejemplito rápido que sirve para repasar 4 conceptos clave de la Clase 06 (estadística y preprocesamiento), porque son el terreno sobre el que se apoya todo lo que viene: para que un modelo de ML "aprenda" de datos (Clase 07), primero hace falta saber **leerlos** con estas mismas herramientas.

Vamos a repasar estos 4 conceptos sobre un dataset real de natalidad, para que el repaso quede conectado con algo real y no sea solo teoría suelta.

### 1) Limpieza e Integración
Ningún dataset real llega listo para analizar. **Limpiar** significa decidir qué hacer con nulos (imputarlos con la media/mediana si son numéricos, con la moda o una etiqueta de negocio si son categóricos) y con duplicados (eliminarlos). La clave es no borrar a ciegas: un nulo puede tener una causa de negocio válida detrás (por ejemplo, un campo que simplemente no aplica a ese registro) y no ser necesariamente un error de carga. **Integrar** significa combinar o derivar columnas nuevas a partir de las existentes, para que la información cruda se vuelva accionable — por ejemplo, transformar un valor numérico en una etiqueta de negocio interpretable.

In [ ]:
import pandas as pd
import numpy as np

# =====================================================================
# REPASO 1: Limpieza e Integración
# =====================================================================
print("=== REPASO 1: LIMPIEZA E INTEGRACIÓN ===")

df_raw = pd.read_csv('tasa-natalidad-deis-2000-2024.csv')
print(f"Filas: {df_raw.shape[0]} (años, de 2000 a 2024) | Columnas: {df_raw.shape[1]} (1 índice de tiempo + 25 provincias)")

nulos = df_raw.isnull().sum().sum()
duplicados = df_raw.duplicated().sum()
print(f"Valores nulos totales: {nulos}")
print(f"Filas duplicadas: {duplicados}")

if nulos == 0 and duplicados == 0:
    print("👉 Este dataset del DEIS ya llega limpio (0 nulos, 0 duplicados).")
    print("   Si hubiera nulos, se imputan con media/mediana (numéricas) o moda/etiqueta de negocio (categóricas);")
    print("   si hubiera duplicados, se eliminan con .drop_duplicates(). El procedimiento es siempre el mismo.")

# Integración: creamos una variable derivada de negocio a partir de un valor numérico
natalidad_2024 = df_raw[df_raw['indice_tiempo'] == '01-01-2024'].drop(columns='indice_tiempo').T
natalidad_2024.columns = ['natalidad_2024']
natalidad_2024['categoria_natalidad'] = pd.cut(
    natalidad_2024['natalidad_2024'],
    bins=[0, 8.4, 9.7, np.inf],
    labels=['Baja', 'Media', 'Alta']
)
print("\nVariable derivada 'categoria_natalidad' (Integración) sobre el último año disponible:")
print(natalidad_2024['categoria_natalidad'].value_counts())

### 2) Medidas de Tendencia Central y Dispersión
La **tendencia central** responde "¿dónde está el centro de los datos?": la **media** (promedio) es sensible a valores extremos, la **mediana** (valor que deja 50% de los datos a cada lado) no lo es, y la **moda** es el valor más frecuente. Cuando media y mediana difieren mucho, es señal de asimetría o de outliers. La **dispersión** responde "¿qué tan esparcidos están?": el **desvío estándar** mide la variación promedio respecto a la media, y el **IQR** (rango intercuartílico, Q3 − Q1) mide el ancho del 50% central de los datos, siendo más robusto frente a outliers.

In [ ]:
# =====================================================================
# REPASO 2: Medidas de Tendencia Central y Dispersión
# =====================================================================
print("=== REPASO 2: TENDENCIA CENTRAL Y DISPERSIÓN ===")

serie_nacional = df_raw['natalidad_argentina']

media = serie_nacional.mean()
mediana = serie_nacional.median()
std = serie_nacional.std()
q1 = serie_nacional.quantile(0.25)
q3 = serie_nacional.quantile(0.75)
iqr = q3 - q1

print(f"Media (natalidad nacional 2000-2024): {media:.2f}")
print(f"Mediana: {mediana:.2f}")
print(f"Desvío estándar: {std:.2f}")
print(f"IQR (Q3 - Q1): {iqr:.2f} (entre {q1:.2f} y {q3:.2f})")

print("\n👉 DEDUCCIÓN: la mediana (17.9) es más alta que la media (16.35).")
print("   Esto no se debe a outliers, sino a que la natalidad viene en caída sostenida:")
print("   hay más años 'altos' (principio de la serie) que 'bajos' (final de la serie),")
print("   por eso el valor central pesa distinto que el promedio. Un IQR de ~3.1 puntos")
print("   confirma que, aun con esa caída, la dispersión año a año es moderada.")

### 3) Distribuciones y Correlación
La **distribución** es la forma que toman los datos al graficarlos (histograma): puede ser simétrica (tipo campana/Normal) o sesgada a la izquierda/derecha. La **correlación** (coeficiente de Pearson, entre -1 y 1) mide qué tan asociadas linealmente están dos variables numéricas: cerca de 1 indica que suben juntas, cerca de -1 que una sube cuando la otra baja, y cerca de 0 que no hay relación lineal. Recordar siempre: **correlación no implica causalidad**.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# =====================================================================
# REPASO 3: Distribuciones y Correlación
# =====================================================================
print("=== REPASO 3: DISTRIBUCIONES Y CORRELACIÓN ===")

plt.figure(figsize=(12, 4))

# Distribución: forma de la serie nacional a lo largo de los 25 años
plt.subplot(1, 2, 1)
sns.histplot(serie_nacional, kde=True, bins=10, color='teal')
plt.axvline(media, color='red', linestyle='--', label=f'Media: {media:.1f}')
plt.axvline(mediana, color='green', linestyle='--', label=f'Mediana: {mediana:.1f}')
plt.title('Distribución de la natalidad nacional (2000-2024)')
plt.xlabel('Tasa de natalidad')
plt.legend()

# Correlación: ¿se mueven juntas las provincias grandes?
plt.subplot(1, 2, 2)
provincias_comparar = df_raw[['natalidad_buenos_aires', 'natalidad_cordoba', 'natalidad_santa_fe']]
matriz_corr = provincias_comparar.corr()
sns.heatmap(matriz_corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, center=0)
plt.title('Correlación entre provincias')

plt.tight_layout()
plt.show()

skew = serie_nacional.skew()
corr_ba_cba = matriz_corr.loc['natalidad_buenos_aires', 'natalidad_cordoba']
print(f"Asimetría (skew) de la serie nacional: {skew:.2f}")
print(f"Correlación Buenos Aires vs. Córdoba: {corr_ba_cba:.2f}")
print("\n👉 DEDUCCIÓN: la correlación es altísima (>0.95) porque ambas provincias comparten la misma")
print("   tendencia demográfica del país (la caída de la natalidad es un fenómeno nacional, no local).")
print("   Ojo: esto es correlación, no causalidad — Córdoba no baja su natalidad *porque* baja Buenos Aires.")

### 4) Transformación y Reducción de Dimensionalidad
Para que un algoritmo matemático procese los datos hace falta **transformarlos**: convertir texto a números (`LabelEncoder`, one-hot encoding) y llevar las variables numéricas a una escala comparable (`StandardScaler`), ya que los modelos basados en distancias (como K-Means, que usaremos hoy) son sensibles a la magnitud de cada columna. Cuando hay muchas columnas, **PCA** (Análisis de Componentes Principales) permite comprimirlas en unas pocas dimensiones que conservan la mayor parte de la variabilidad original, simplificando el análisis sin perder información relevante.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# =====================================================================
# REPASO 4: Transformación y Reducción de Dimensionalidad
# =====================================================================
print("=== REPASO 4: TRANSFORMACIÓN Y REDUCCIÓN ===")

# Transformación: escalado de un par de columnas para que queden comparables
columnas_ejemplo = df_raw[['natalidad_buenos_aires', 'natalidad_cordoba']]
scaler_demo = StandardScaler()
columnas_escaladas = scaler_demo.fit_transform(columnas_ejemplo)

print("Antes de escalar (media original):")
print(columnas_ejemplo.mean().round(2).to_dict())
print("Después de escalar con StandardScaler (media ≈ 0, desvío ≈ 1):")
print(pd.DataFrame(columnas_escaladas, columns=columnas_ejemplo.columns).describe().loc[['mean', 'std']].round(2))

# Reducción: PCA sobre todas las provincias (25 años = 25 features por provincia)
provincias_T = df_raw.drop(columns='indice_tiempo').T
X_scaled_demo = StandardScaler().fit_transform(provincias_T)
pca_demo = PCA(n_components=2)
pca_demo.fit_transform(X_scaled_demo)
varianza_total = pca_demo.explained_variance_ratio_.sum() * 100

print(f"\nVarianza explicada al comprimir 25 años en 2 componentes principales: {varianza_total:.1f}%")
print("👉 DEDUCCIÓN: escalar es obligatorio antes de PCA o K-Means porque estos algoritmos miden distancias;")
print("   sin escalar, una columna con números más grandes 'pesaría' más en el resultado solo por su magnitud,")
print("   no porque sea más importante. Este mismo criterio (dividir train/test, ajustar el escalador solo con train)")
print("   lo vamos a retomar formalmente en la Clase 07, con train_test_split de Scikit-Learn.")

# 🤖 Semana 8 — Aplicaciones Prácticas de ML
### Clase práctica: del algoritmo al impacto real

**Dataset:** `propiedades_sueca_ml.csv` — la continuación de la "valija" de la Semana 7, ya limpia y lista para entrenar un primer modelo.

**Recorrido de hoy:**
1. El mapa de la IA/ML/DL y los tipos de aprendizaje
2. Scikit-Learn por dentro: estimators, transformers y predictors
3. Entrenar y evaluar sin trampas: train/test y sobreajuste
4. Consolidación: del modelo al ciclo de vida completo

In [ ]:
# Setup inicial
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score

sns.set_style("whitegrid")

df = pd.read_csv("propiedades_sueca_ml.csv")
df.head(10)

---
## 🗺️ Bloque 1 — El mapa de la IA, ML y DL (0:00–0:30)

### Rompehielo
Antes de una sola definición: miremos 5 filas del dataset, SIN la columna precio.

**Pregunta para el grupo:** si tuviera que escribir un programa con reglas fijas ("SI superficie > 100 Y barrio == Centro, ENTONCES precio > 200.000") para estimar el precio de estas propiedades, ¿cuántas reglas creen que necesitaría? ¿Alcanzaría alguna vez?

In [ ]:
# Rompehielo: features sin la etiqueta (precio)
df.drop(columns=["precio_eur", "precio_por_m2", "id_propiedad"]).sample(5, random_state=1)

### El mapa completo: IA → ML → DL
Como las muñecas matrioskas: **Inteligencia Artificial** es la más grande, **Machine Learning** está adentro, y **Deep Learning** es la más chica de todas, adentro del ML.

- **IA:** cualquier sistema que imite tareas que requieren inteligencia humana (incluye reglas fijas "SI-ENTONCES").
- **ML:** en vez de programar reglas, le damos ejemplos (datos) y el algoritmo encuentra el patrón.
- **DL:** un tipo de ML con redes neuronales de muchas capas, ideal para datos no estructurados (imágenes, audio, texto libre).

**Para hoy:** vamos a trabajar 100% en la capa de Machine Learning clásico — la que domina cuando los datos son tablas, como la nuestra.

### Tipos de aprendizaje
Con el mismo dataset de propiedades, tres formas distintas de aprender:

- **Supervisado:** tenemos la etiqueta (`precio_eur`). El modelo aprende la relación entre features y precio. Es lo que vamos a hacer hoy.
- **No supervisado:** si borráramos `precio_eur`, todavía podríamos agrupar propiedades parecidas entre sí (clustering) — sin que nadie les diga a qué grupo pertenecen. Lo vamos a ver en la Semana 10.
- **Por refuerzo:** un agente que prueba y se equivoca, ganando o perdiendo "puntos" — como un sistema de precios dinámicos que ajusta tarifas y aprende de la reacción del mercado.

**Pregunta para el grupo:** ¿supervisado, no supervisado o por refuerzo?
- Agrupar clientes de un supermercado por hábitos de compra, sin categorías previas.
- Predecir si un mail es spam, usando miles de mails ya marcados.
- Un termostato inteligente que aprende a ahorrar energía probando distintas temperaturas.

---
## ⚙️ Bloque 2 — Scikit-Learn por dentro (0:30–1:00)

### Estimators, Transformers y Predictors
Toda la librería sigue la misma lógica, sin importar el algoritmo: **instanciar → fit → transform/predict**.

Primero, dividimos el dataset en train y test — nunca entrenamos con todos los datos.

In [ ]:
features = ["superficie_m2", "ambientes", "antiguedad_anios", "score_amenities"]
X = df[features]
y = df["precio_eur"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Filas de entrenamiento:", X_train.shape[0])
print("Filas de prueba:", X_test.shape[0])

### Transformer en acción: StandardScaler
Un Transformer "estudia" los datos con `.fit()` y después los modifica con `.transform()`. Lo vimos la semana pasada como concepto — ahora lo vemos como objeto de Scikit-Learn.

⚠️ Regla de oro: el `.fit()` se hace SOLO con el train. El test se transforma con lo que el scaler ya aprendió, nunca vuelve a aprender de él.

In [ ]:
scaler = StandardScaler()

# fit_transform en train: aprende (fit) y aplica (transform) en un solo paso
X_train_scaled = scaler.fit_transform(X_train)

# SOLO transform en test: aplica lo ya aprendido, no vuelve a estudiar
X_test_scaled = scaler.transform(X_test)

print("Media aprendida por columna:", scaler.mean_.round(1))
print("Forma de X_train_scaled:", X_train_scaled.shape)

### Estimator + Predictor en acción: LinearRegression
Para este primer modelo NO vamos a escalar los features — con regresión lineal simple no hace falta, y así los coeficientes quedan directamente interpretables en euros. (Sí haría falta escalar con KNN o regresión regularizada — lo vamos a ver en la Semana 9.)

In [ ]:
modelo = LinearRegression()

# fit: el modelo "estudia" la relación entre features y precio
modelo.fit(X_train, y_train)

# predict: usamos lo aprendido para predecir sobre datos nuevos (test)
predicciones = modelo.predict(X_test)

print("Primeras 5 predicciones:", predicciones[:5].round(0))
print("Primeros 5 precios reales:", y_test.values[:5])

### Atributos aprendidos (terminan en `_`)
`coef_` nos dice cuánto pesa cada feature en la predicción — y con estos datos SIN escalar, se interpreta directo en euros.

In [ ]:
coeficientes = pd.Series(modelo.coef_, index=features).round(1)
print(coeficientes)
print("\nIntercepto (precio base):", round(modelo.intercept_))

**Para leer con el grupo:** por cada metro cuadrado adicional, el precio sube en promedio esa cantidad de euros, manteniendo todo lo demás constante. La antigüedad debería restar valor — revisemos el signo. Y `score_amenities` (cochera + balcón) suma directo al precio.

---
## 🎯 Bloque 3 — Entrenar y evaluar sin trampas (1:10–1:35)

### Métricas: ¿qué tan bueno es el modelo?
Dos métricas típicas para regresión: **R²** (0 a 1, cuánta variación explica el modelo) y **MAE** (error promedio, en las mismas unidades que el precio — euros).

In [ ]:
pred_train = modelo.predict(X_train)
pred_test = modelo.predict(X_test)

print("=== Regresión lineal ===")
print(f"R² train: {r2_score(y_train, pred_train):.3f}   R² test: {r2_score(y_test, pred_test):.3f}")
print(f"MAE train: {mean_absolute_error(y_train, pred_train):,.0f} €   MAE test: {mean_absolute_error(y_test, pred_test):,.0f} €")

**Para discutir:** train y test dan valores parecidos — es señal de que el modelo generalizó bien, no memorizó. Ahora vamos a ver a propósito lo contrario: un modelo que sí memoriza.

### Overfitting en acción: árbol de decisión sin restricciones
Un árbol de decisión sin límite de profundidad puede crecer hasta memorizar cada fila del train.

**Antes de correr la celda, arriesguen:** ¿qué esperan que pase con el R² de train? ¿Y con el de test?

In [ ]:
arbol_libre = DecisionTreeRegressor(random_state=42)  # sin max_depth: crece sin límite
arbol_libre.fit(X_train, y_train)

print("=== Árbol SIN restricciones ===")
print(f"R² train: {r2_score(y_train, arbol_libre.predict(X_train)):.3f}   R² test: {r2_score(y_test, arbol_libre.predict(X_test)):.3f}")
print(f"MAE train: {mean_absolute_error(y_train, arbol_libre.predict(X_train)):,.0f} €   MAE test: {mean_absolute_error(y_test, arbol_libre.predict(X_test)):,.0f} €")

¡Ahí está! R² de train prácticamente perfecto (memorizó), pero el error en test es mucho más alto que el de la regresión lineal. Es el "estudiante que se memoriza las respuestas": perfecto en la guía de ejercicios, se derrumba en el examen real.

### La solución: limitar la complejidad

In [ ]:
arbol_limitado = DecisionTreeRegressor(max_depth=4, random_state=42)
arbol_limitado.fit(X_train, y_train)

print("=== Árbol CON max_depth=4 ===")
print(f"R² train: {r2_score(y_train, arbol_limitado.predict(X_train)):.3f}   R² test: {r2_score(y_test, arbol_limitado.predict(X_test)):.3f}")
print(f"MAE train: {mean_absolute_error(y_train, arbol_limitado.predict(X_train)):,.0f} €   MAE test: {mean_absolute_error(y_test, arbol_limitado.predict(X_test)):,.0f} €")

**Para el pizarrón:** comparemos los 3 modelos — regresión lineal, árbol libre y árbol limitado — anotando el "gap" (diferencia) entre R² de train y test de cada uno. Cuanto más grande el gap, más sobreajuste.

In [ ]:
comparacion = pd.DataFrame({
    "Modelo": ["Regresión lineal", "Árbol SIN restricción", "Árbol max_depth=4"],
    "R2_train": [
        r2_score(y_train, pred_train),
        r2_score(y_train, arbol_libre.predict(X_train)),
        r2_score(y_train, arbol_limitado.predict(X_train)),
    ],
    "R2_test": [
        r2_score(y_test, pred_test),
        r2_score(y_test, arbol_libre.predict(X_test)),
        r2_score(y_test, arbol_limitado.predict(X_test)),
    ],
})
comparacion["gap"] = (comparacion["R2_train"] - comparacion["R2_test"]).round(3)
comparacion.round(3)

### 🐛 La trampa del data leakage
En el dataset tenemos una columna `precio_por_m2` — que ya vimos en la Semana 7. ¿Qué pasa si la usamos como feature para predecir `precio_eur`?

In [ ]:
# ERROR A PROPÓSITO: precio_por_m2 se calculó A PARTIR de precio_eur
features_con_leakage = features + ["precio_por_m2"]
X_leak = df[features_con_leakage]

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leak, y, test_size=0.2, random_state=42)

modelo_leak = LinearRegression()
modelo_leak.fit(X_train_l, y_train_l)

print("R² test CON data leakage:", round(r2_score(y_test_l, modelo_leak.predict(X_test_l)), 4))

**¿Sospechoso, no?** Un R² casi perfecto no es motivo de festejo — es la primera señal de alarma. `precio_por_m2` se calculó dividiendo `precio_eur` por `superficie_m2`, así que le estamos dando al modelo una pista que contiene casi la respuesta. En la vida real, ese dato ni siquiera existiría todavía al momento de predecir el precio de una propiedad nueva.

---
## 🔗 Bloque 4 — Consolidación guiada (1:35–1:55)

### El ciclo de vida completo de un proyecto de ML
Entrenar el modelo (lo que hicimos hoy) es solo una etapa. El ciclo completo:

1. **Definición del problema** — ¿clasificación o regresión? (hoy: regresión de precio)
2. **Datos** — la valija que armamos la semana pasada
3. **Entrenamiento** — lo que acabamos de hacer con `.fit()`
4. **Evaluación** — comparar train vs test, detectar overfitting
5. **Despliegue (inferencia)** — `.predict()` sobre propiedades nuevas, reales
6. **Monitoreo** — el mundo cambia; el modelo puede "desactualizarse" con el tiempo

Vamos a resolver 4 tareas juntos, en plenario, para cerrar la clase.

In [ ]:
# Tarea 2 (guiada): interpretar los coeficientes en términos de negocio
coeficientes.sort_values(ascending=False)

In [ ]:
# Tarea 3 (guiada): diagnosticar overfitting a partir de números dados
casos = pd.DataFrame({
    "caso": ["Modelo A", "Modelo B", "Modelo C"],
    "R2_train": [0.95, 0.55, 0.99],
    "R2_test": [0.93, 0.52, 0.61],
})
casos["gap"] = (casos["R2_train"] - casos["R2_test"]).round(2)
casos

**Para resolver en vivo con el grupo:** ¿cuál de los tres casos (A, B o C) muestra overfitting? ¿Cuál muestra underfitting? ¿Cuál es el "sweet spot"?

---
## ✅ Cierre — De la teoría a la pre-entrega

Hoy entrenamos, evaluamos y detectamos overfitting con un modelo real. En la pre-entrega de esta semana no vas a programar un modelo — vas a **pensar como Data Scientist**: identificar una oportunidad de ML en tu propio entorno, definir qué datos necesitarías y qué sesgos podrían aparecer.

### 🎧 Para reforzar
Escuchen el **PODCAST — Aplicaciones Prácticas de ML** de la semana como repaso antes de encarar la pre-entrega.

---
## 🔑 Solucionario — Resolución del ejercicio guiado (uso docente)

> Para tener a mano mientras conducís las 4 tareas del Bloque 4 en plenario.

### Tarea 1 — Tipos de aprendizaje (respuestas)
- Agrupar clientes sin categorías previas → **No supervisado** (clustering)
- Predecir spam con mails ya marcados → **Supervisado** (clasificación)
- Termostato que prueba y aprende de la reacción → **Por refuerzo**

In [ ]:
# Tarea 2 — ya resuelta arriba con coeficientes.sort_values()
# Lectura esperada: superficie_m2 y score_amenities suman valor: antiguedad_anios resta.
# ambientes puede dar un coeficiente chico o levemente negativo controlando por superficie
# (multicolinealidad: una propiedad más grande ya "trae" más ambientes).
coeficientes.sort_values(ascending=False)

### Tarea 3 — Diagnóstico (respuestas)
- **Modelo A** (R² train 0.95, test 0.93, gap 0.02) → **Sweet spot**. Generaliza bien.
- **Modelo B** (R² train 0.55, test 0.52, gap 0.03) → **Underfitting**. Gap chico, pero el error es alto en ambos — el modelo es demasiado simple para el problema.
- **Modelo C** (R² train 0.99, test 0.61, gap 0.38) → **Overfitting**. Memorizó el train, no generaliza.

### Tarea 4 — Próximo paso ante overfitting (consigna abierta)
No hay una única respuesta — opciones válidas que puede proponer el grupo:
- Reducir la complejidad del modelo (bajar `max_depth`, menos features)
- Conseguir más datos de entrenamiento
- Regularización (Ridge/Lasso — se ve en la Semana 9)
- Validación cruzada para elegir mejor los hiperparámetros
- Eliminar features irrelevantes o correlacionadas entre sí